In [5]:
# Use the full path: LakehouseName.TableName
df = spark.read.table("Suppy_BronzeLakeHouse.dbo.Inventory_SupplyChain_Dataset")

# Check shape (rows & columns)
print("Rows:", df.count())
print("Columns:", len(df.columns))

# Check column names and data types
df.printSchema()

# Basic statistics
display(df.describe())



StatementMeta(, ecb0ac27-84e4-401b-a3a7-eab7f91b568e, 7, Finished, Available, Finished, False)

Rows: 1200
Columns: 15
root
 |-- Date: date (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Supplier: string (nullable = true)
 |-- Warehouse: string (nullable = true)
 |-- Order Status: string (nullable = true)
 |-- Units Sold: long (nullable = true)
 |-- Inventory Level: long (nullable = true)
 |-- Transportation Cost: double (nullable = true)
 |-- Order Accuracy: boolean (nullable = true)
 |-- Lead Time _Days_: long (nullable = true)
 |-- Backorder: boolean (nullable = true)
 |-- Cost of Goods Sold _COGS_: double (nullable = true)
 |-- Average Inventory: double (nullable = true)
 |-- Warehouse Capacity: long (nullable = true)



SynapseWidget(Synapse.DataFrame, f9de9963-fa6a-4ea8-aa9d-92f2fc08a9cd)

### Handle Missing / Null Values

In [7]:
from pyspark.sql.functions import col, sum as spark_sum

# Check nulls in each column
null_counts = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df.columns
])
display(null_counts)

# Option A: Drop rows with ANY null values
df_cleaned = df.dropna()




StatementMeta(, ecb0ac27-84e4-401b-a3a7-eab7f91b568e, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 78b39f5d-9a94-4663-bb0a-20c003b54541)

Rename & Fix Column Names

In [8]:

# Clean all column names (remove spaces, lowercase)
from pyspark.sql import functions as F

new_columns = [c.strip().lower().replace(" ", "_") for c in df_cleaned.columns]
df_cleaned = df_cleaned.toDF(*new_columns)

df_cleaned.printSchema()


StatementMeta(, ecb0ac27-84e4-401b-a3a7-eab7f91b568e, 10, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- supplier: string (nullable = true)
 |-- warehouse: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- units_sold: long (nullable = true)
 |-- inventory_level: long (nullable = true)
 |-- transportation_cost: double (nullable = true)
 |-- order_accuracy: boolean (nullable = true)
 |-- lead_time__days_: long (nullable = true)
 |-- backorder: boolean (nullable = true)
 |-- cost_of_goods_sold__cogs_: double (nullable = true)
 |-- average_inventory: double (nullable = true)
 |-- warehouse_capacity: long (nullable = true)



Fix Data Types

### 

In [13]:
from pyspark.sql.functions import col

# 1. Cleaning & Renaming (Removing spaces for Gold standard)
df_cleaned =df_cleaned.withColumnRenamed("cost_of_goods_sold__cogs_", "COGS")
df_cleaned.printSchema()
            
             



StatementMeta(, ecb0ac27-84e4-401b-a3a7-eab7f91b568e, 15, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- category: string (nullable = true)
 |-- supplier: string (nullable = true)
 |-- warehouse: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- units_sold: long (nullable = true)
 |-- inventory_level: long (nullable = true)
 |-- transportation_cost: double (nullable = true)
 |-- order_accuracy: boolean (nullable = true)
 |-- lead_time__days_: long (nullable = true)
 |-- backorder: boolean (nullable = true)
 |-- COGS: double (nullable = true)
 |-- average_inventory: double (nullable = true)
 |-- warehouse_capacity: long (nullable = true)



Save Cleaned Data Back to Lakehouse

In [14]:
# 2. Save it to your Silver Lakehouse

df_cleaned.write.format("delta").mode("overwrite").saveAsTable("Cleaned_Inventory_SupplyChain_Dataset")

print("Transformation Complete! Your clean data is now in Silver.")

StatementMeta(, ecb0ac27-84e4-401b-a3a7-eab7f91b568e, 16, Finished, Available, Finished, False)

Transformation Complete! Your clean data is now in Silver.


### 